# 02. Arguments

**Tier:** Intermediate. **Why you'd use this:** A cart function should add an item to the caller's list. A total function should accept any number of prices. A greeting should work with no name, using a default.

**After this lesson you can:** grow the caller's list through a parameter; collect extra prices with `*args`; explain why a default is baked in at `def` time.
About 25 minutes. **Uses:** [Definitions](01-definitions-lesson.ipynb)


## Quick review

1. What does a function return when it never hits `return`?
2. Why does `list(grid)` still share a nested row?

<details><summary>Answers</summary>

It returns `None`. A shallow copy copies pointers, so the row objects stay shared.
</details>


## The idea

Arguments are passed by sharing. The parameter name points at the same object the caller passed. Mutating that object is visible to the caller. Rebinding the parameter name is not.

A default fills in an argument the caller left out. That default expression runs once, when `def` runs, not on every call. `*args` packs extra positional arguments into a tuple.

<details><summary>Explain more</summary>

`def add(a, b=0)` means `add(2)` is `add(2, 0)`. Inside `show(*args)`, the name `args` is an ordinary tuple. Assigning `items = items + [value]` makes a new list and points the parameter at it. The caller's name still points at the old list.
</details>


## See it work

Add an item to a cart the caller already has, and total any number of prices.


In [1]:
def add_item(cart, item):
    # 1. Mutate the list the caller passed.
    cart.append(item)
    return cart


def total(*prices):
    # 2. prices is every positional argument, as a tuple.
    amount = 0
    for price in prices:
        amount = amount + price
    return amount


basket = ["bread"]
print(add_item(basket, "milk"), basket)
print(total(3, 4, 5), total())

['bread', 'milk'] ['bread', 'milk']
12 0


## Check yourself

Answer these before you run the cell.

1. After `extend_wrong`, does `nums` contain `2`?
2. What does `mark()` print on the second call, and why isn't it a new list each time?


In [2]:
def extend_wrong(items, value):
    items = items + [value]
    return items


nums = [1]
print(extend_wrong(nums, 2), nums)


def mark(seen=[]):
    seen.append("x")
    return seen


print(mark())
print(mark())

[1, 2] [1]
['x']
['x', 'x']


<details><summary>Answers</summary>

`extend_wrong` returns `[1, 2]`, but `nums` is still `[1]`. The parameter was rebound to a new list. `mark` prints `['x']` then `['x', 'x']`. The default list was created once at `def` time, so every call shares it.
</details>


## Why it works this way

A parameter slot receives a pointer. `cart.append(item)` follows that pointer and changes the list object. Every name that points at that list sees the change. `items = items + [value]` builds a new list and writes a new pointer into the parameter slot only. The caller's name is a different slot and still holds the old pointer. That is call by sharing: mutation is shared, rebinding is local. Deep dive: [Mutability and aliasing](../../internals/02-mutability/01-aliasing-lesson.ipynb).

A default is evaluated when the `def` statement runs. The object it produces is stored on the function, which is why `once.__defaults__` still holds `0` after `stamp` becomes `9`. Later calls reuse that stored object when the argument is omitted. A fresh empty list as a default is therefore one list for every call that skips the argument. Appends accumulate across calls. Numbers and strings are safe defaults because you do not mutate them in place. A mutable bag is not.

The cost of `*args` is a new tuple of the extra pointers. Packing does not copy the argument objects. The design keeps defaults cheap and shared on purpose. Rebuilding a new default object on every call would hide the mutable-default bug and make every call allocate. Sharing the default is the cheaper rule, and the rule you must remember when the default is a list or dict.

Evidence for "once" is the `stamp` cell: the default is the integer that existed at `def` time. Changing the global afterward does not rewrite `__defaults__`. Evidence for sharing is the `mark` cell: two calls append into one list. Together they show that defaults are objects with identity, not templates that rerun.


In [3]:
stamp = 0


def once(n=stamp):
    return n


stamp = 9
print(once(), once())
print(once.__defaults__)

0 0
(0,)


## When to use it

Mutate a parameter when the caller wants one list updated. Return a new list when the caller must keep the old one. Use `*args` for any number of like values. Use a default for an optional number or string. Don't use a mutable default. Build a new list inside the body when the caller omits the argument.


## Common mistake

Rebinding the parameter and thinking the caller's list changed. The returned list has the new item. The name the caller still holds does not.


In [4]:
def add_item_wrong(cart, item):
    cart = cart + [item]
    return cart


basket = ["bread"]
print(add_item_wrong(basket, "milk"))
print(basket)

['bread', 'milk']
['bread']


`append` changes the object both names share. The return can even be omitted when the caller already has the list, but returning it keeps the call useful in an expression.


In [5]:
def add_item(cart, item):
    cart.append(item)
    return cart


basket = ["bread"]
print(add_item(basket, "milk"))
print(basket)

['bread', 'milk']
['bread', 'milk']


## Try it

Change `total` so a missing call is `None` instead of `0`, by giving one optional argument with a default of `None` and treating that as empty. Why must that default not be a list you append prices into?


## Recap

- Parameters share objects with the caller. Mutate to share a change. Rebind to keep the caller unchanged.
- Defaults are evaluated once, at `def` time.
- `*args` collects extra positional arguments into a tuple.

**Next:** [practice](02-arguments-practice.ipynb)
